 Install missing libraries

In [ ]:

!pip install vaderSentiment feedparser yfinance -q

print("Done! All libraries ready.")

1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import feedparser
import warnings, os
from datetime import datetime, timedelta
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import copy

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = "/kaggle/working/"
os.makedirs(OUT, exist_ok=True)

CONFIG = {
    "tickers":    ["AAPL", "TSLA", "MSFT", "AMZN", "GOOGL"],
    "start_date": "2022-01-01",
    "end_date":   "2024-01-01",
}
print(f"Device: {DEVICE} | Tickers: {CONFIG['tickers']}")


 2. Data Collection & Merging

In [ ]:
# ── 1. Stock prices (Yahoo Finance) ─────────────────────────
def compute_rsi(s, p=14):
    d = s.diff()
    g = d.clip(lower=0).rolling(p).mean()
    l = (-d.clip(upper=0)).rolling(p).mean()
    return 100 - (100 / (1 + g / (l + 1e-8)))

def compute_bollinger(s, p=20):
    ma = s.rolling(p).mean()
    std = s.rolling(p).std()
    upper = ma + 2*std
    lower = ma - 2*std
    return upper, lower, (s - lower) / (upper - lower + 1e-8)  # %B

def get_stocks(cfg):
    frames = []
    for t in cfg["tickers"]:
        df = yf.download(t, start=cfg["start_date"], end=cfg["end_date"], progress=False)
        df.reset_index(inplace=True)
        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
        df["Ticker"] = t
        df["Next_Close"]       = df["Close"].shift(-1)
        df["Market_Direction"] = (df["Next_Close"] > df["Close"]).astype(int)
        df["Pct_Change"]       = df["Close"].pct_change() * 100
        df["Volatility"]       = df["Pct_Change"].rolling(5).std()
        df["MA_7"]             = df["Close"].rolling(7).mean()
        df["MA_21"]            = df["Close"].rolling(21).mean()
        df["MA_50"]            = df["Close"].rolling(50).mean()           # NEW
        df["RSI"]              = compute_rsi(df["Close"])
        df["RSI_14"]           = compute_rsi(df["Close"], 14)             # explicit
        df["MACD"]             = df["Close"].ewm(span=12).mean() - df["Close"].ewm(span=26).mean()
        df["MACD_signal"]      = df["MACD"].ewm(span=9).mean()            # NEW
        df["MACD_hist"]        = df["MACD"] - df["MACD_signal"]           # NEW
        bb_up, bb_low, bb_pct  = compute_bollinger(df["Close"])
        df["BB_upper"]         = bb_up                                    # NEW
        df["BB_lower"]         = bb_low                                   # NEW
        df["BB_pct"]           = bb_pct                                   # NEW
        df["OBV"]              = (np.sign(df["Close"].diff()) * df["Volume"]).cumsum()  # NEW
        df["Price_MA7_ratio"]  = df["Close"] / (df["MA_7"] + 1e-8)        # NEW
        df["Price_MA21_ratio"] = df["Close"] / (df["MA_21"] + 1e-8)       # NEW
        frames.append(df)
    out = pd.concat(frames, ignore_index=True).dropna(subset=["Next_Close"])
    out["Date"] = pd.to_datetime(out["Date"]).dt.date
    print(f"Stocks: {out.shape}")
    return out

stock_df = get_stocks(CONFIG)

# ── 2. News sentiment (Reuters RSS) ────────────────────────
analyzer = SentimentIntensityAnalyzer()

def rss_sentiment(urls):
    rows = []
    for url in urls:
        feed = feedparser.parse(url)
        for e in feed.entries:
            pub = e.get("published_parsed") or e.get("updated_parsed")
            date = datetime(*pub[:3]).date() if pub else datetime.today().date()
            score = analyzer.polarity_scores(e.get("title",""))["compound"]
            label = "positive" if score>=0.05 else ("negative" if score<=-0.05 else "neutral")
            rows.append({"Date": date, "headline": e.get("title",""),
                         "vader_score": score, "vader_label": label})
    return pd.DataFrame(rows)

rss_urls = [
    "https://feeds.finance.yahoo.com/rss/2.0/headline?s=AAPL&region=US&lang=en-US",
    "https://feeds.finance.yahoo.com/rss/2.0/headline?s=TSLA&region=US&lang=en-US",
    "http://feeds.reuters.com/reuters/businessNews",
]
rss_df = rss_sentiment(rss_urls)
print(f"RSS headlines: {len(rss_df)}")

# ── 3. Kaggle dataset ───────────────────────────────────────
import glob
matches = glob.glob("/kaggle/input/**/all-data.csv", recursive=True) + \
          glob.glob("/kaggle/input/**/*.csv", recursive=True)

KAGGLE_PATH = "/kaggle/input/datasets/ankurzing/sentiment-analysis-for-financial-news/all-data.csv"
print(f"Found dataset at: {KAGGLE_PATH}")

if KAGGLE_PATH and os.path.exists(KAGGLE_PATH):
    kg = pd.read_csv(KAGGLE_PATH, encoding="latin-1", header=None,
                     names=["label","headline"])
    kg["label"] = kg["label"].str.strip().str.lower()
    kg = kg[kg["label"].isin(["positive","negative","neutral"])]
    dates = pd.date_range(CONFIG["start_date"], CONFIG["end_date"], freq="B")
    kg["Date"] = np.random.choice(dates, size=len(kg))
    kg["Date"] = pd.to_datetime(kg["Date"]).dt.date
    score_fn = lambda t: analyzer.polarity_scores(str(t))["compound"]
    kg["vader_score"] = kg["headline"].apply(score_fn)
    kg["vader_label"] = kg["vader_score"].apply(
        lambda s: "positive" if s>=0.05 else ("negative" if s<=-0.05 else "neutral"))
    kaggle_df = kg[["Date","headline","vader_score","vader_label"]]
    print(f"Kaggle rows: {len(kaggle_df)}")
else:
    kaggle_df = pd.DataFrame(columns=["Date","headline","vader_score","vader_label"])
    print("Kaggle dataset not found — using RSS only.")

# ── 4. Aggregate daily sentiment ───────────────────────────
all_news = pd.concat([rss_df, kaggle_df], ignore_index=True)
all_news["Date"] = pd.to_datetime(all_news["Date"]).dt.date

agg = all_news.groupby("Date").agg(
    news_count     =("vader_score","count"),
    avg_sentiment  =("vader_score","mean"),
    positive_count =("vader_label", lambda x:(x=="positive").sum()),
    negative_count =("vader_label", lambda x:(x=="negative").sum()),
).reset_index()
total = agg["news_count"].replace(0,1)
agg["positive_ratio"] = agg["positive_count"] / total
agg["negative_ratio"] = agg["negative_count"] / total
agg["sentiment_momentum"] = agg["avg_sentiment"].rolling(3, min_periods=1).mean()
print(f"Daily sentiment rows: {len(agg)}")

# ── 5. Merge ────────────────────────────────────────────────
stock_df["Date"] = pd.to_datetime(stock_df["Date"]).dt.date
agg["Date"]      = pd.to_datetime(agg["Date"]).dt.date
merged = pd.merge(stock_df, agg, on="Date", how="left").fillna(
    {"news_count":0,"avg_sentiment":0,"positive_count":0,
     "negative_count":0,"positive_ratio":0,"negative_ratio":0,
     "sentiment_momentum":0}).dropna()

merged.to_csv(OUT+"final_dataset.csv", index=False)
print(f"Final merged dataset: {merged.shape}")
merged.head()


3. EDA (Exploratory Data Analysis)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Market Movement Dataset — EDA", fontsize=15, fontweight="bold")

# 1. Class balance
merged["Market_Direction"].value_counts().plot(
    kind="bar", ax=axes[0,0], color=["#e74c3c","#2ecc71"], edgecolor="black")
axes[0,0].set_title("Market Direction Distribution")
axes[0,0].set_xticklabels(["Down (0)","Up (1)"], rotation=0)

# 2. Sentiment over time (AAPL)
aapl = merged[merged["Ticker"]=="AAPL"].sort_values("Date")
axes[0,1].plot(pd.to_datetime(aapl["Date"]), aapl["avg_sentiment"], color="#3498db", lw=0.8)
axes[0,1].axhline(0, color="red", ls="--", lw=0.8)
axes[0,1].set_title("Daily Avg Sentiment (AAPL)")

# 3. Close price
for t in merged["Ticker"].unique():
    sub = merged[merged["Ticker"]==t].sort_values("Date")
    axes[0,2].plot(pd.to_datetime(sub["Date"]), sub["Close"], label=t, lw=0.8)
axes[0,2].set_title("Close Price Over Time")
axes[0,2].legend(fontsize=7)

# 4. Volatility
axes[1,0].plot(pd.to_datetime(aapl["Date"]), aapl["Volatility"], color="#e67e22", lw=0.8)
axes[1,0].set_title("Volatility Over Time (AAPL)")

# 5. Sentiment vs direction
sent_dir = merged.groupby(["avg_sentiment","Market_Direction"]).size().reset_index()
axes[1,1].scatter(merged["avg_sentiment"], merged["Market_Direction"],
                  alpha=0.3, color="#9b59b6", s=10)
axes[1,1].set_title("Sentiment vs Market Direction")
axes[1,1].set_xlabel("Avg Sentiment"); axes[1,1].set_ylabel("Direction (0/1)")

# 6. Correlation heatmap
num_cols = ["Close","Volume","avg_sentiment","positive_ratio",
            "negative_ratio","RSI","MACD","Volatility","Market_Direction"]
sns.heatmap(merged[num_cols].corr(), ax=axes[1,2], cmap="coolwarm",
            annot=True, fmt=".2f",  annot_kws={"size": 7})
axes[1,2].set_title("Correlation Heatmap")

plt.tight_layout()
plt.savefig(OUT+"eda.png", dpi=150, bbox_inches="tight")
plt.show()
print("EDA saved!")

 4. Sequence Builder + Model Training

In [15]:
# ═══════════════════════════════════════════════════════════════
#  HYPERPARAMETER SEARCH  ──  finds best BATCH & EPOCHS for you
# ═══════════════════════════════════════════════════════════════
FEATURE_COLS = ["Open","High","Low","Close","Volume","MA_7","MA_21",
                "RSI","MACD","Volatility","avg_sentiment","positive_ratio",
                "negative_ratio","sentiment_momentum","news_count"]
TARGET  = "Market_Direction"
SEQ_LEN = 10
LR      = 0.001

# ── Candidate values to search ───────────────────────────────
BATCH_CANDIDATES  = [32, 64, 128]   # small / medium / large
EPOCH_CANDIDATES  = [30, 60, 100]   # short / medium / long
MAX_PATIENCE      = 10              # early-stop patience (epochs)

# ── Build sequences (done once) ──────────────────────────────
df = pd.read_csv(OUT+"final_dataset.csv")
df = df.sort_values(["Ticker","Date"]).reset_index(drop=True)
scaler = MinMaxScaler()
df[FEATURE_COLS] = scaler.fit_transform(df[FEATURE_COLS])

X_list, y_list = [], []
for ticker in df["Ticker"].unique():
    sub = df[df["Ticker"]==ticker].reset_index(drop=True)
    F = sub[FEATURE_COLS].values; L = sub[TARGET].values
    for i in range(len(sub)-SEQ_LEN):
        X_list.append(F[i:i+SEQ_LEN]); y_list.append(L[i+SEQ_LEN])

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.int64)
print(f"Sequences: X={X.shape}  y={y.shape}")

X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=0.15, shuffle=False)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.176, shuffle=False)

class TSD(Dataset):
    def __init__(self,X,y): self.X=torch.tensor(X); self.y=torch.tensor(y)
    def __len__(self): return len(self.X)
    def __getitem__(self,i): return self.X[i], self.y[i]

test_dl = DataLoader(TSD(X_test, y_test), batch_size=64)

IN, HID, NL, DO, NC = len(FEATURE_COLS), 128, 2, 0.3, 2

class RNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(IN,HID,NL,batch_first=True,dropout=DO)
        self.bn=nn.BatchNorm1d(HID); self.drop=nn.Dropout(DO); self.fc=nn.Linear(HID,NC)
    def forward(self,x):
        o,_=self.rnn(x); return self.fc(self.drop(self.bn(o[:,-1,:])))

class LSTMModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(IN,HID,NL,batch_first=True,dropout=DO)
        self.bn=nn.BatchNorm1d(HID); self.drop=nn.Dropout(DO); self.fc=nn.Linear(HID,NC)
    def forward(self,x):
        o,_=self.lstm(x); return self.fc(self.drop(self.bn(o[:,-1,:])))

class GRUModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru=nn.GRU(IN,HID,NL,batch_first=True,dropout=DO)
        self.bn=nn.BatchNorm1d(HID); self.drop=nn.Dropout(DO); self.fc=nn.Linear(HID,NC)
    def forward(self,x):
        o,_=self.gru(x); return self.fc(self.drop(self.bn(o[:,-1,:])))

MODEL_CLASSES = {"RNN": RNNModel, "LSTM": LSTMModel, "GRU": GRUModel}

# ── Core train function ───────────────────────────────────────
def train_once(model_cls, batch_size, max_epochs, patience=MAX_PATIENCE):
    """Train one model with given batch_size/max_epochs.
       Returns (model, best_val_acc, stopped_at_epoch, history)."""
    train_dl = DataLoader(TSD(X_train,y_train), batch_size=batch_size, shuffle=True)
    val_dl   = DataLoader(TSD(X_val,  y_val),   batch_size=batch_size)

    model = model_cls().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    crit  = nn.CrossEntropyLoss()
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)

    best_val_acc, best_w, pat, stopped_ep = 0.0, None, 0, max_epochs
    hist = {"ta":[], "va":[], "tl":[], "vl":[]}

    for ep in range(1, max_epochs+1):
        # ── train ──
        model.train(); tl=tc=tt=0
        for Xb,yb in train_dl:
            Xb,yb = Xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad(); out=model(Xb); loss=crit(out,yb)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
            tl+=loss.item()*len(yb); tc+=(out.argmax(1)==yb).sum().item(); tt+=len(yb)
        # ── validate ──
        model.eval(); vl=vc=vt=0
        with torch.no_grad():
            for Xb,yb in val_dl:
                Xb,yb=Xb.to(DEVICE),yb.to(DEVICE); out=model(Xb); loss=crit(out,yb)
                vl+=loss.item()*len(yb); vc+=(out.argmax(1)==yb).sum().item(); vt+=len(yb)
        val_acc = vc/vt
        hist["ta"].append(tc/tt); hist["va"].append(val_acc)
        hist["tl"].append(tl/tt); hist["vl"].append(vl/vt)
        sched.step(vl/vt)

        # ── early stop on best val accuracy ──
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_w = copy.deepcopy(model.state_dict())
            pat = 0
        else:
            pat += 1
            if pat >= patience:
                stopped_ep = ep
                break

    model.load_state_dict(best_w)
    return model, best_val_acc, stopped_ep, hist

# ══════════════════════════════════════════════════════════════
#  GRID SEARCH  ──  try every (batch, epoch) combo per model
# ══════════════════════════════════════════════════════════════
search_results = {}   # {model_name: {(batch,epoch): val_acc}}
best_configs   = {}   # {model_name: {"batch","epochs","val_acc","model","hist"}}

total = len(MODEL_CLASSES) * len(BATCH_CANDIDATES) * len(EPOCH_CANDIDATES)
done  = 0

for mname, mcls in MODEL_CLASSES.items():
    search_results[mname] = {}
    best_acc = 0.0

    for batch in BATCH_CANDIDATES:
        for max_ep in EPOCH_CANDIDATES:
            done += 1
            print(f"[{done}/{total}] {mname}  batch={batch}  max_epochs={max_ep}  ...", end=" ")
            torch.manual_seed(0); np.random.seed(0)
            model, val_acc, stopped, hist = train_once(mcls, batch, max_ep)
            search_results[mname][(batch, max_ep)] = val_acc
            print(f"val_acc={val_acc:.4f}  stopped@ep{stopped}")

            if val_acc > best_acc:
                best_acc = val_acc
                best_configs[mname] = {
                    "batch": batch, "epochs": max_ep,
                    "val_acc": val_acc, "stopped_ep": stopped,
                    "model": model, "hist": hist
                }

# ── Save .pth files to disk ───────────────────────────────────
for mname, cfg in best_configs.items():
    path = OUT + f"model_{mname.lower()}.pth"
    torch.save(cfg["model"].state_dict(), path)
    print(f"💾 Saved: model_{mname.lower()}.pth — {os.path.getsize(path)/1024:.0f} KB")
print("\n✅ Grid search complete!")
print("\n" + "="*60)
print(f"  {'Model':<10} {'Best Batch':>12} {'Best Epochs':>13} {'Stopped@':>10} {'Val Acc':>9}")
print("-"*60)
for mname, cfg in best_configs.items():
    print(f"  {mname:<10} {cfg['batch']:>12} {cfg['epochs']:>13} "
          f"ep{cfg['stopped_ep']:>7} {cfg['val_acc']:>9.4f}")


Sequences: X=(2205, 10, 15)  y=(2205,)
[1/27] RNN  batch=32  max_epochs=30  ... val_acc=0.5364  stopped@ep14
[2/27] RNN  batch=32  max_epochs=60  ... val_acc=0.5364  stopped@ep14
[3/27] RNN  batch=32  max_epochs=100  ... val_acc=0.5364  stopped@ep14
[4/27] RNN  batch=64  max_epochs=30  ... val_acc=0.5606  stopped@ep12
[5/27] RNN  batch=64  max_epochs=60  ... val_acc=0.5606  stopped@ep12
[6/27] RNN  batch=64  max_epochs=100  ... val_acc=0.5606  stopped@ep12
[7/27] RNN  batch=128  max_epochs=30  ... val_acc=0.5606  stopped@ep15
[8/27] RNN  batch=128  max_epochs=60  ... val_acc=0.5606  stopped@ep15
[9/27] RNN  batch=128  max_epochs=100  ... val_acc=0.5606  stopped@ep15
[10/27] LSTM  batch=32  max_epochs=30  ... val_acc=0.5333  stopped@ep16
[11/27] LSTM  batch=32  max_epochs=60  ... val_acc=0.5333  stopped@ep16
[12/27] LSTM  batch=32  max_epochs=100  ... val_acc=0.5333  stopped@ep16
[13/27] LSTM  batch=64  max_epochs=30  ... val_acc=0.5394  stopped@ep12
[14/27] LSTM  batch=64  max_epochs=6

5. Evaluation & Comparison Plots

5.1: Evaluate

In [ ]:
trained   = {n: best_configs[n]["model"]  for n in best_configs}
histories = {n: best_configs[n]["hist"]   for n in best_configs}

def evaluate(model, name):
    model.eval(); preds, labels = [], []
    with torch.no_grad():
        for Xb,yb in test_dl:
            p = model(Xb.to(DEVICE)).argmax(1).cpu().numpy()
            preds.extend(p); labels.extend(yb.numpy())
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="weighted")
    cm  = confusion_matrix(labels, preds)
    cfg = best_configs[name]
    print(f"\n{name}  [batch={cfg['batch']} epochs={cfg['epochs']} stopped@ep{cfg['stopped_ep']}]")
    print(f"  Accuracy: {acc:.4f} | F1: {f1:.4f}")
    print(classification_report(labels, preds, target_names=["Down","Up"]))
    return {"acc":acc, "f1":f1, "cm":cm, "preds":preds, "labels":labels}

results = {n: evaluate(m, n) for n, m in trained.items()}


5.2 Training curves

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(22,9))
fig.suptitle("Training Curves — All Models", fontsize=14, fontweight="bold")
for col, (name, h) in enumerate(histories.items()):
    ep = range(1, len(h["tl"])+1)
    axes[0,col].plot(ep, h["tl"], label="Train", color="#3498db")
    axes[0,col].plot(ep, h["vl"], label="Val",   color="#e74c3c")
    axes[0,col].set_title(f"{name} — Loss"); axes[0,col].legend()
    axes[1,col].plot(ep, h["ta"], label="Train", color="#2ecc71")
    axes[1,col].plot(ep, h["va"], label="Val",   color="#e67e22")
    axes[1,col].set_title(f"{name} — Accuracy"); axes[1,col].legend()
plt.tight_layout()
plt.savefig(OUT+"training_curves.png", dpi=150, bbox_inches="tight")
plt.show()



 5.3 Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22,5))
fig.suptitle("Confusion Matrices", fontsize=14, fontweight="bold")
for col, (name, r) in enumerate(results.items()):
    sns.heatmap(r["cm"], annot=True, fmt="d", cmap="Blues",
                xticklabels=["Down","Up"], yticklabels=["Down","Up"], ax=axes[col])
    axes[col].set_title(f"{name}\nAcc={r['acc']:.3f} F1={r['f1']:.3f}")
plt.tight_layout()
plt.savefig(OUT+"confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()



5.4 Comparison bar chart

In [ ]:
names = list(results.keys())
accs  = [results[n]["acc"] for n in names]
f1s   = [results[n]["f1"]  for n in names]
x = np.arange(len(names)); w = 0.35
fig, ax = plt.subplots(figsize=(9,5))
ax.bar(x-w/2, accs, w, label="Accuracy", color="#3498db", edgecolor="black")
ax.bar(x+w/2, f1s,  w, label="F1-Score", color="#2ecc71", edgecolor="black")
ax.set_xticks(x); ax.set_xticklabels(names, fontsize=12)
ax.set_ylim(0,1.1); ax.legend()
ax.set_title("Model Comparison", fontsize=13)
for i,(a,f) in enumerate(zip(accs,f1s)):
    ax.text(i-w/2, a+0.02, f"{a:.3f}", ha="center", fontsize=9)
    ax.text(i+w/2, f+0.02, f"{f:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(OUT+"model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()



5.5 Final Summary

In [ ]:
print("\n" + "="*60)
print(f"  {'Model':<10} {'Batch':>7} {'Epochs':>8} {'Stopped':>9} {'Accuracy':>10} {'F1':>8}")
print("-"*60)
names = list(results.keys())
for n in names:
    cfg = best_configs[n]
    marker = "  ◀ BEST" if n == max(names, key=lambda x: results[x]["acc"]) else ""
    print(f"  {n:<10} {cfg['batch']:>7} {cfg['epochs']:>8} "
          f"ep{cfg['stopped_ep']:>6} {results[n]['acc']:>10.4f} "
          f"{results[n]['f1']:>8.4f}{marker}")
best = max(names, key=lambda n: results[n]["acc"])
print(f"\n  Best overall: {best}")
print(f"  Batch={best_configs[best]['batch']}  "
      f"Epochs={best_configs[best]['epochs']}  "
      f"Naturally stopped @ ep{best_configs[best]['stopped_ep']}")
print("\n  ℹ  Early stopping fired before max_epochs → no need to")
print("     increase epochs further. These ARE the optimal values.")
